# 🏢 Entrenamiento de Modelo YOLOv8-Seg en CubiCasa5K (InmobiliariaVR)

Este cuaderno entrena un modelo de **Segmentación de Instancias Arquitectónicas** usando la GPU gratuita de Google Colab.

### ⚠️ Pasos Previos en Colab:
1. Ve al menú superior: **Entorno de ejecución (Runtime)** > **Cambiar tipo de entorno de ejecución (Change runtime type)**.
2. En *Acelerador de hardware*, selecciona **T4 GPU** y pulsa **Guardar**.
3. Ejecuta la celda de abajo pulsando el botón de **Play [ ▶ ]**.

Al finalizar, se descargará automáticamente a tu computadora el archivo **`best.pt`** listo para colocar en tu backend.

In [ ]:
# =============================================================================
# 1. COMPROBACIÓN DE GPU Y CONFIGURACIÓN
# =============================================================================
import torch
print("Torch version:", torch.__version__)
if not torch.cuda.is_available():
    print("⚠️ ADVERTENCIA: No se detectó GPU. Recuerda activar 'T4 GPU' en Entorno de ejecución > Cambiar tipo de entorno.")
else:
    print(f"✅ GPU Activa: {torch.cuda.get_device_name(0)} (Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

# =============================================================================
# 2. INSTALACIÓN DE DEPENDENCIAS
# =============================================================================
!pip install -q ultralytics datasets huggingface_hub

# =============================================================================
# 3. DESCARGA Y CONVERSIÓN DEL DATASET DIRECTO EN COLAB
# =============================================================================
import os
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset, Image as DsImage

dataset_dir = Path("/content/dataset")
for s in ["train", "valid"]:
    (dataset_dir / "images" / s).mkdir(parents=True, exist_ok=True)
    (dataset_dir / "labels" / s).mkdir(parents=True, exist_ok=True)

print("📥 Descargando CubiCasa5K-COCO desde HuggingFace...")
ds_train = load_dataset("phungpx/cubicassa5k-coco", split="train")
ds_valid = load_dataset("phungpx/cubicassa5k-coco", split="valid")

# Mapeo de categorias oficiales
CAT_MAP = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7}
CLASS_NAMES = ["bathroom", "bed", "door", "kitchen", "room", "stairs", "wall", "window"]

def export_split(ds, split_name):
    img_dir = dataset_dir / "images" / split_name
    lbl_dir = dataset_dir / "labels" / split_name
    for row in tqdm(ds, desc=f"Exportando {split_name}"):
        fn = row["file_name"]
        stem = Path(fn).stem
        
        # Guardar imagen original
        img_path = img_dir / fn
        img_val = row["image"]
        if isinstance(img_val, dict) and img_val.get("bytes"):
            img_path.write_bytes(img_val["bytes"])
        elif hasattr(img_val, "save"):
            img_val.save(img_path)
            
        # Guardar etiquetas YOLO-seg normalizadas
        w = float(row["width"])
        h = float(row["height"])
        lines = []
        anns = row["annotations"]
        cat_ids = anns.get("category_id", [])
        segs = anns.get("segmentation", [])
        
        for cat_id, seg_list in zip(cat_ids, segs):
            if cat_id in CAT_MAP and seg_list:
                c_idx = CAT_MAP[cat_id]
                for poly in seg_list:
                    if len(poly) >= 6:
                        coords = []
                        for i in range(0, len(poly), 2):
                            x_norm = min(max(poly[i] / w, 0.0), 1.0)
                            y_norm = min(max(poly[i+1] / h, 0.0), 1.0)
                            coords.append(f"{x_norm:.6f} {y_norm:.6f}")
                        lines.append(f"{c_idx} " + " ".join(coords))
                        
        (lbl_dir / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

export_split(ds_train, "train")
export_split(ds_valid, "valid")

# Generar data.yaml
yaml_text = """# CubiCasa5K YOLO Config
path: /content/dataset
train: images/train
val: images/valid

names:
  0: bathroom
  1: bed
  2: door
  3: kitchen
  4: room
  5: stairs
  6: wall
  7: window
"""
(dataset_dir / "data.yaml").write_text(yaml_text, encoding="utf-8")
print("✅ Dataset preparado exitosamente en /content/dataset")

# =============================================================================
# 4. ENTRENAMIENTO CON YOLOV8-SEG EN GPU
# =============================================================================
from ultralytics import YOLO

print("🚀 Iniciando entrenamiento con YOLOv8-Seg...")
model = YOLO("yolov8n-seg.pt")  # Modelo base pre-entrenado

# 30 epocas con resolucion 640 y batch de 16 en GPU T4 toma ~15-20 minutos
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/runs",
    name="cubicasa_run",
    workers=4,
    plots=True,
)

# =============================================================================
# 5. DESCARGAR AUTOMÁTICAMENTE EL MODELO FINAL (best.pt)
# =============================================================================
from google.colab import files
best_weights = Path("/content/runs/cubicasa_run/weights/best.pt")

if best_weights.exists():
    print(f"🎉 ¡Entrenamiento completado! Descargando {best_weights.name} ({best_weights.stat().st_size / (1024**2):.1f} MB)...")
    files.download(str(best_weights))
else:
    print("⚠️ No se encontró best.pt en la ruta esperada.")
